# Embeddingi perfum

**Cel:** dla każdych z ok. 24 tys. perfum policzyć wektor, w którym **bliskość (cosinus) = podobieństwo zapachu**. Wynik trafia do kolejnego etapu (wizualizacja, klasteryzacja) i do bazy aplikacji.

**Wejście:** `fra_cleaned_final.csv`. **Wyjście:** `Z_final.npy` (macierz 24 046 × 512, ta sama kolejność wierszy co w CSV) oraz plik z identyfikatorami perfum.

**Jak to działa:**
1. Każde perfumy zamieniamy na liczby: wektor nut (jedna współrzędna na nutę) i wektor akordów (waga maleje z pozycją akordu: 1.0, 0.75, 0.56, 0.42, 0.32 – kolejność akordów jest informacją).
2. **Metoda A** (TF-IDF + SVD) – klasyczna, trzyma wyraźne akordy i „braci” z jednej linii zapachowej.
3. **Metoda B\*** (uczenie wektorów nut i akordów z ich współwystępowania + uśrednianie) – lepiej przewiduje zapach z samych nut.
4. **Wynik C\*** = sklejenie A (30%) i B\* (70%).

**Wagi warstw piramidy (głowa/serce/baza) = 1/1/1** – ustalone w poprzednich badaniach.
**Nie scalamy tu nut szczegółowych w ogólne** (zrobisz to później przy budowie bazy – patrz podsumowanie na końcu).

**Trzy miary, tylko tyle potrzeba do decyzji:**

| Miara | Co mówi | Kierunek |
|---|---|---|
| **Zgodność akordów (zimny start)** | czy sąsiedzi perfum mają podobne akordy, gdy model znał **tylko nuty** perfum zapytania (akordy zapytania były przed modelem ukryte) | wyżej = lepiej; losowo ≈ 0,23 |
| **Bracia w top-10** | jaka część wariantów tej samej linii (ta sama marka i początek nazwy) trafia do 10 najbliższych | wyżej = lepiej |
| **Skośność hubness** | czy niektóre perfumy nie „zaśmiecają” cudzych list sąsiadów | niżej = lepiej (ok. 2–3 to normalnie) |

## 1. Ustawienia i dane

**Co robi kod.** Importuje biblioteki, zbiera wszystkie parametry w jednym słowniku `CONFIG` (wartości wybrane w poprzednich badaniach, opis w komentarzach), wczytuje CSV i zamienia teksty na listy:
* `Top/Middle/Base` – listy nut (małe litery, bez „unknown”),
* `accords_l` – lista do 5 akordów **w kolejności ważności**,
* `perfume_id` – numer z adresu url (`...-93845.html` → 93845); to stały klucz perfum, niezależny od kolejności wierszy (przyda się w bazie).

`N` to liczba perfum – używana w całym notatniku.

In [1]:
import json, os, re, time, warnings
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import skew
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.utils.extmath import randomized_svd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)

CONFIG = dict(
    OUTPUT_DIR = "wyniki_embeddingow",
    MIN_DF_NOTE = 3,                                      # nuta musi wystąpić w >= 3 perfumach (rzadsze to szum)
    ACCORD_RANK_WEIGHTS = [1.0, 0.75, 0.56, 0.42, 0.32],  # waga akordu wg pozycji 1.-5.
    IDF_POWER = 0.5,                                      # osłabianie pospolitych cech (0 = brak, 1 = pełne)
    A_ALPHA_ACCORD = 0.4,                                 # metoda A: udział akordów w podobieństwie
    B_CTX_SMOOTH = 0.5,                                   # metoda B: wygładzenie rozkładu kontekstów w PPMI
    B_ACCORD_GAIN = 1.5,                                  # metoda B: mnożnik wagi akordów przy uśrednianiu
    B_TOP1_WEIGHT = 0.75,                                 # metoda B*: waga tokena "akord nr 1"
    B_PPMI_SHIFT = 0.69,                                  # metoda B*: przesunięcie PMI (ok. ln 2)
    HYBRID_W_A = 0.30,                                    # udział metody A w wyniku C*
    DIM = 256,                                            # długość wektora każdej metody (wynik: 2 x 256 = 512)
    SEED = 42,
)
INPUT_CSV = "../dane/fra_cleaned_final.csv"
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

df = pd.read_csv(INPUT_CSV, sep=";", encoding="utf-8-sig")
N = len(df)

def split_list(s):
    if not isinstance(s, str):
        return []
    return [x.strip().lower() for x in s.split(",") if x.strip() and x.strip().lower() != "unknown"]

for warstwa in ["Top", "Middle", "Base"]:
    df[warstwa + "_l"] = df[warstwa].map(split_list)
df["accords_l"] = df[[f"mainaccord{i}" for i in range(1, 6)]].apply(
    lambda row: [str(x).strip().lower() for x in row if isinstance(x, str) and x.strip()], axis=1)
df["perfume_id"] = df.url.str.extract(r"-(\d+)\.html$")[0].astype("Int64")

print("perfum:", N, "| unikalne perfume_id:", df.perfume_id.nunique(), "| brak id:", int(df.perfume_id.isna().sum()))
print("unikalne nuty:", len(Counter(n for l in ["Top_l", "Middle_l", "Base_l"] for lst in df[l] for n in lst)),
      "| unikalne akordy:", len(Counter(a for lst in df.accords_l for a in lst)))

perfum: 24046 | unikalne perfume_id: 24046 | brak id: 0
unikalne nuty: 1664 | unikalne akordy: 84


## 2. Wektory cech (nuty i akordy jako macierze rzadkie)

**Teoria.** Perfumy mają po kilkanaście nut z ponad tysiąca możliwych, więc wektor cech to same zera z kilkunastoma wartościami – dlatego trzymamy go jako **macierz rzadką** (zapamiętuje tylko niezerowe elementy).
* **Nuty:** waga 1 za każde wystąpienie (nuta powtórzona w kilku warstwach dostaje sumę).
* **Akordy:** waga zależy od pozycji (`ACCORD_RANK_WEIGHTS`).
* **`MIN_DF_NOTE`:** nuty z mniej niż 3 perfum są pomijane (szum). Kod sprawdza, czy żadne perfumy nie zostały bez nut.

**Co robi kod:** `to_sparse` zamienia listę słowników `{cecha: waga}` na macierz CSR (wiersze = perfumy, kolumny = cechy). Wynik: `X_note` (nuty) i `X_acc` (akordy), z nazwami kolumn `vocab_note`, `vocab_acc`.

In [2]:
def to_sparse(cechy, min_df=1):
    licznik = Counter(k for d in cechy for k in d)                       # w ilu perfumach jest cecha
    vocab = sorted(k for k, v in licznik.items() if v >= min_df)
    indeks = {k: i for i, k in enumerate(vocab)}
    r, c, v = [], [], []
    for i, d in enumerate(cechy):
        for k, w in d.items():
            j = indeks.get(k)
            if j is not None:
                r.append(i); c.append(j); v.append(w)
    return sp.csr_matrix((v, (r, c)), shape=(len(cechy), len(vocab))), vocab

def note_features():
    wynik = []
    for lists in zip(df.Top_l, df.Middle_l, df.Base_l):                  # wagi warstw 1/1/1
        d = {}
        for lst in lists:
            for n in lst:
                d[n] = d.get(n, 0.0) + 1.0
        wynik.append(d)
    return wynik

wagi = CONFIG["ACCORD_RANK_WEIGHTS"]
X_note, vocab_note = to_sparse(note_features(), CONFIG["MIN_DF_NOTE"])
X_acc,  vocab_acc  = to_sparse([{a: wagi[i] for i, a in enumerate(l[:5])} for l in df.accords_l])

bez_nut = int((np.asarray(X_note.sum(axis=1)).ravel() == 0).sum())
print("tabela nut:", X_note.shape, "| tabela akordów:", X_acc.shape, "| perfumy bez żadnej nuty po filtrze:", bez_nut)
assert bez_nut == 0, "zmniejsz MIN_DF_NOTE"

tabela nut: (24046, 1133) | tabela akordów: (24046, 84) | perfumy bez żadnej nuty po filtrze: 0


## 3. Dwie metody budowy wektorów i ich połączenie

### Metoda A – TF-IDF + SVD
1. **IDF:** pospolite cechy (piżmo jest w połowie perfum) mówią mało o *różnicy* między perfumami, więc ważymy cechę przez `IDF^0,5`, gdzie `IDF = ln((1+N)/(1+df)) + 1`.
2. **Dwa bloki:** wektor nut i wektor akordów skracamy każdy do długości 1 i sklejamy: `[√α·akordy, √(1−α)·nuty]`. Dzięki pierwiastkom iloczyn takich wektorów to dokładnie `α·cos(akordy) + (1−α)·cos(nuty)`.
3. **SVD:** kompresja z ok. 1100 do 256 liczb – podobne nuty (np. różne cytrusy) trafiają na te same osie.

### Metoda B\* – wektory nut i akordów uczone z ich współwystępowania
1. Każda nuta (`n:vanilla`) i akord (`a:woody`) to „słowo”, każde perfumy to „zdanie”. Dokładamy też słowo `a1:<akord>` dla akordu **nr 1** (dominującego).
2. Liczymy, w ilu perfumach każda para słów występuje razem, i zamieniamy to na **PPMI** – o ile częściej niż przypadkiem: `log P(i,j) / (P(i)·P(j))`, tylko wartości dodatnie. Przesunięcie `B_PPMI_SHIFT` odejmuje od PMI stałą przed obcięciem do zera – wycina słabe, przypadkowe powiązania.
3. SVD tej tabeli daje wektor dla każdej nuty i akordu (podobne nuty – podobne wektory).
4. **Wektor perfum = średnia wektorów jej nut i akordów** (akordy ważone pozycją × `B_ACCORD_GAIN`); na końcu usuwamy wspólny kierunek wszystkich perfum, który zawyżałby cosinusy.

### Połączenie
`hybrid(A, B, w)` skleja `[√w·A, √(1−w)·B]`, więc podobieństwo = `w·cos_A + (1−w)·cos_B` – średnia „opinii” obu metod.

**Co robi kod:** `build_A`, `build_B`, `hybrid`. Funkcje biorą macierze jako argumenty (a nie zmienne globalne) – to potrzebne w następnym kroku, bo do uczciwej oceny budujemy model na danych z ukrytymi akordami. `build_B(top1_weight=0, ppmi_shift=0)` to starsza wersja B bez ulepszeń (używana do porównania).

In [3]:
def idf_vec(X, power):
    dfc = np.asarray((X > 0).sum(axis=0)).ravel()
    return (np.log((1 + X.shape[0]) / (1 + dfc)) + 1.0) ** power

def build_A(Xn, Xa):
    a, p = CONFIG["A_ALPHA_ACCORD"], CONFIG["IDF_POWER"]
    Sn = normalize(Xn.multiply(idf_vec(Xn, p)).tocsr())
    Sa = normalize(Xa.multiply(idf_vec(Xa, p)).tocsr())
    V = sp.hstack([np.sqrt(a) * Sa, np.sqrt(1 - a) * Sn]).tocsr()
    return normalize(TruncatedSVD(CONFIG["DIM"], random_state=CONFIG["SEED"]).fit_transform(V)).astype(np.float32)

def build_B(Xn, Xa, top1_weight=None, ppmi_shift=None):
    top1_weight = CONFIG["B_TOP1_WEIGHT"] if top1_weight is None else top1_weight
    ppmi_shift = CONFIG["B_PPMI_SHIFT"] if ppmi_shift is None else ppmi_shift
    bloki = [Xn, Xa * CONFIG["B_ACCORD_GAIN"]]
    if top1_weight > 0:                                                  # tokeny "akord nr 1"
        bloki.append((Xa >= wagi[0] - 1e-9).astype(float) * top1_weight)
    W = sp.hstack(bloki).tocsr()
    Wb = W.copy(); Wb.data[:] = 1                                        # liczymy samą obecność
    C = (Wb.T @ Wb).toarray().astype(np.float64)                         # C[i,j] = w ilu perfumach razem i oraz j
    np.fill_diagonal(C, 0)
    p_i = C.sum(axis=1) / C.sum()
    p_j = C.sum(axis=0) ** CONFIG["B_CTX_SMOOTH"]; p_j /= p_j.sum()      # wygładzony rozkład kontekstów
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log((C / C.sum()) / (p_i[:, None] * p_j[None, :])) - ppmi_shift
    ppmi = np.where(np.isfinite(pmi) & (pmi > 0), pmi, 0.0)
    U, _, _ = randomized_svd(sp.csr_matrix(ppmi), n_components=CONFIG["DIM"], random_state=CONFIG["SEED"])
    P = normalize(normalize(W) @ normalize(U))                           # średnia wektorów tokenów
    u = TruncatedSVD(1, random_state=CONFIG["SEED"]).fit(P).components_[0]
    return normalize(P - np.outer(P @ u, u)).astype(np.float32)          # usuń wspólny kierunek

def hybrid(Za, Zb, w_A):
    return normalize(np.hstack([np.sqrt(w_A) * Za, np.sqrt(1 - w_A) * Zb])).astype(np.float32)

Z_A = build_A(X_note, X_acc)
Z_Bs = build_B(X_note, X_acc)
Z_final = hybrid(Z_A, Z_Bs, CONFIG["HYBRID_W_A"])
print("A:", Z_A.shape, "| B*:", Z_Bs.shape, "| C* (wynik końcowy):", Z_final.shape)

A: (24046, 256) | B*: (24046, 256) | C* (wynik końcowy): (24046, 512)


## 4. Ocena modeli (trzy miary)

**Problem.** Nie mamy etykiet „te perfumy są podobne”. Jedyny sygnał o odbieranym zapachu to **akordy**. Pytamy więc: *czy sąsiedzi perfum mają podobne akordy?* (cosinus wektorów akordów, uśredniony po 10 sąsiadach).

**Pułapka.** Jeśli akordy są częścią wektora, test sprawdza tylko, czy model zapamiętał to, co mu wpisano (wynik ok. 0,8 niezależnie od jakości). Dlatego stosujemy **zimny start**: perfumy dzielimy na 5 części; dla każdej części **usuwamy akordy jej perfum jeszcze przed dopasowaniem modelu**, pytamy o ich sąsiadów (znamy tylko nuty) i oceniamy po prawdziwych akordach. Kandydatami na sąsiadów są pozostałe perfumy.

**Co robi kod:**
* `knn_top` – dla porcji zapytań liczy cosinusy do wszystkich kandydatów (iloczyn macierzy) i wybiera 10 najbliższych; remisy rozstrzyga deterministycznie (wynik jest powtarzalny).
* `ocena(model_fn)` – (1) zimny start: zwraca średnią zgodność akordów ± błąd standardowy; (2) na modelu dopasowanym na całości liczy „braci w top-10” i skośność hubness. Funkcja `model_fn(X_note, X_acc)` buduje wektory z podanych macierzy.

Błąd standardowy (SE) = odchylenie / √n; różnice mniejsze niż ok. 2–3 SE traktuj jako szum.

In [4]:
Y = normalize(X_acc).toarray().astype(np.float32)          # "odpowiedzi": pełne akordy z wagami rangowymi
FLANK = pd.factorize(df.Brand + "|" + df.Perfume.str.split().str[:2].str.join(" "))[0]   # klucz "braci"
LOSOWO = float(Y.mean(0) @ Y.mean(0))

def knn_top(Zq, Zdb, q, k=10, kandydaci=None, chunk=1000):
    Zq, Zdb = np.asarray(Zq, np.float32), np.asarray(Zdb, np.float32)
    out = np.zeros((len(q), k), dtype=np.int64)
    for s in range(0, len(q), chunk):
        qi = q[s:s + chunk]
        sim = Zq[s:s + chunk] @ Zdb.T
        if kandydaci is not None:
            sim[:, ~kandydaci] = -9
        sim[np.arange(len(qi)), qi] = -9                                 # perfumy nie są własnym sąsiadem
        czesc = np.argpartition(-sim, k + 40, axis=1)[:, :k + 40]
        czesc.sort(axis=1)                                               # remisy: niższy indeks wygrywa
        kol = np.argsort(-np.take_along_axis(sim, czesc, axis=1), axis=1, kind="stable")[:, :k]
        out[s:s + chunk] = np.take_along_axis(czesc, kol, axis=1)
    return out

def zgodnosc_akordow(top, q):                              # AccAgree@10 na zapytanie
    return np.einsum("qkd,qd->qk", Y[top], Y[q]).mean(1)

def ocena(model_fn, n_folds=5):
    fold = np.random.default_rng(CONFIG["SEED"]).permutation(N) % n_folds
    wyniki = []
    for f in range(n_folds):                                           # ZIMNY START
        H = fold == f
        Xa_ukryte = (sp.diags((~H).astype(float)) @ X_acc).tocsr()      # akordy perfum z H usunięte przed modelem
        Z = model_fn(X_note, Xa_ukryte)
        q = np.where(H)[0]
        wyniki.append(zgodnosc_akordow(knn_top(Z[q], Z, q, kandydaci=~H), q))
    acc = np.concatenate(wyniki)

    Z = model_fn(X_note, X_acc)                                        # model na całości: bracia i hubness
    q = np.arange(N)
    top = knn_top(Z, Z, q)
    grp = np.bincount(FLANK)[FLANK]; ok = grp > 1
    bracia = float(((FLANK[top] == FLANK[:, None]).sum(1)[ok] / np.minimum(10, grp[ok] - 1)).mean())
    nk = np.bincount(top.ravel(), minlength=N)
    return dict(acc=acc, bracia=bracia, hub=float(skew(nk)), anty=float((nk == 0).mean()))

print(f"poziom losowy zgodności akordów: {LOSOWO:.3f}")

poziom losowy zgodności akordów: 0.232


## 5. Porównanie: A, B (starsza), B\* i C\*

**Co zobaczysz:** dla każdego modelu zgodność akordów w zimnym starcie (± SE), odsetek „braci” w top-10, skośność hubness i odsetek perfum, które nie trafiają do żadnej cudzej listy (anty-huby). Pod tabelą – różnice parowane (te same zapytania w obu modelach), z błędem standardowym.

**Jak czytać:**
* **B\*** powinno wyraźnie wygrać z B w zgodności akordów (token „akord nr 1” i przesunięty PPMI).
* **A** jest najsłabsze w zgodności, ale ma najwięcej „braci”.
* **C\*** = kompromis: zgodność prawie jak B\*, więcej braci niż B\*, najniższy hubness.

Ostrzeżenie: parametry dobrano na tym samym zbiorze i tej samej mierze, więc rzeczywista przewaga może być nieco mniejsza. Ostateczny sprawdzian to Twoje pary podobieństwa (`pairs_test.csv`).

In [5]:
modele = {
    "A":                    lambda Xn, Xa: build_A(Xn, Xa),
    "B (starsza)":          lambda Xn, Xa: build_B(Xn, Xa, top1_weight=0.0, ppmi_shift=0.0),
    "B*":                   lambda Xn, Xa: build_B(Xn, Xa),
    "C* (A 30% + B* 70%)":  lambda Xn, Xa: hybrid(build_A(Xn, Xa), build_B(Xn, Xa), CONFIG["HYBRID_W_A"]),
}
wyniki, t0 = {}, time.time()
for nazwa, fn in modele.items():
    wyniki[nazwa] = ocena(fn)
    print(f"  policzono {nazwa} ({time.time() - t0:.0f} s)")

tab = pd.DataFrame([{"model": n, "zgodność akordów (zimny start)": r["acc"].mean(),
                     "SE": r["acc"].std(ddof=1) / np.sqrt(len(r["acc"])),
                     "bracia w top-10": r["bracia"], "skośność hubness": r["hub"], "anty-huby": r["anty"]}
                    for n, r in wyniki.items()])
display(tab.round(4))

print("\nróżnice parowane zgodności akordów (średnia ± SE):")
for a, b in [("B*", "B (starsza)"), ("C* (A 30% + B* 70%)", "A"), ("C* (A 30% + B* 70%)", "B*")]:
    d = wyniki[a]["acc"] - wyniki[b]["acc"]
    print(f"  {a} minus {b}: {d.mean():+.4f} ± {d.std(ddof=1) / np.sqrt(len(d)):.4f}")

  policzono A (10 s)
  policzono B (starsza) (18 s)
  policzono B* (26 s)
  policzono C* (A 30% + B* 70%) (39 s)


,model,zgodność akordów (zimny start),SE,bracia w top-10,skośność hubness,anty-huby
0,A,0.5175,0.0009,0.1522,2.5872,0.0271
1,B (starsza),0.5831,0.0011,0.1283,2.8972,0.1031
2,B*,0.6336,0.0011,0.1120,2.4814,0.0893
3,C* (A 30% + B* 70%),0.6287,0.0010,0.1343,2.2397,0.0589



różnice parowane zgodności akordów (średnia ± SE):
  B* minus B (starsza): +0.0504 ± 0.0005
  C* (A 30% + B* 70%) minus A: +0.1113 ± 0.0007
  C* (A 30% + B* 70%) minus B*: -0.0048 ± 0.0003


## 6. Kontrola „na oko”: sąsiedzi znanych perfum

**Cel:** liczby nie zastąpią spojrzenia na konkretne zapachy. Dla kilku znanych perfum wypisujemy 5 najbliższych sąsiadów z wyniku końcowego C\* (z pominięciem tej samej marki – żeby „bracia” nie zaśmiecali listy) razem z trzema pierwszymi akordami. Oceń, czy to ta sama **rodzina zapachowa**.

**Co robi kod:** `find` znajduje wiersz po nazwie i fragmencie marki (nazwy w pliku są wielkimi literami; przy kilku wpisach bierze pierwszy i ostrzega). Listę `TEST_PERFUMES` możesz rozszerzać.

In [6]:
def find(nazwa, marka=None):
    m = df[df.Perfume == nazwa.upper()]
    if marka:
        m = m[m.Brand.str.contains(marka.upper(), regex=False)]
    if len(m) > 1:
        print(f"  (uwaga: {len(m)} wpisów '{nazwa}', biorę pierwszy)")
    return int(m.index[0]) if len(m) else None

def sasiedzi(i, Z, k=5):
    sim = Z @ Z[i]
    sim[i] = -9
    sim[(df.Brand == df.Brand[i]).values] = -9
    return np.argsort(-sim, kind="stable")[:k]

TEST_PERFUMES = [("AVENTUS", "CREED"), ("ANGEL", "MUGLER"), ("LIGHT BLUE", "DOLCE"), ("TUSCAN LEATHER", "TOM FORD"),
                 ("ALIEN", "MUGLER"), ("BABYLON", "PENHALIGON")]
for nazwa, marka in TEST_PERFUMES:
    i = find(nazwa, marka)
    if i is None:
        print("brak:", nazwa); continue
    print(f"\n{df.Perfume[i]} ({df.Brand[i]}) [{', '.join(df.accords_l[i][:3])}]")
    for j in sasiedzi(i, Z_final):
        print(f"    {df.Perfume[j][:30]:30s} {df.Brand[j][:18]:18s} [{', '.join(df.accords_l[j][:3])}]")


AVENTUS (CREED) [fruity, sweet, leather]
    SUPREMACY SILVER               AFNAN              [fruity, leather, woody]
    INVINCIBILE                    TONINO LAMBORGHINI [fruity, woody, leather]
    MATAR                          SPIRIT OF KINGS    [fruity, woody, sweet]
    SHAWQ                          SWISS ARABIAN      [fruity, sweet, woody]
    PINEAPPLE VINTAGE NOIR         PARFUMS VINTAGE    [fruity, leather, woody]

ANGEL (MUGLER) [sweet, patchouli, fruity]
    RIVER OF LOVE                  LA RIVE            [sweet, fruity, warm spicy]
    SILVER LIGHT                   LINN YOUNG         [sweet, patchouli, caramel]
    ELIANA GLAMOUR                 JEQUITI            [sweet, fruity, caramel]
    B U TEASE BLUE                 SARANTIS           [sweet, fruity, patchouli]
    BLUE SPIRIT                    PARIS ELYSEES      [sweet, patchouli, woody]

LIGHT BLUE (DOLCE GABBANA) [citrus, woody, fresh]
    OCEANA BLUE                    GIORGIO MONTI      [woody, citrus,

## 7. Zapis wyników

**Co zapisujemy** (folder `OUTPUT_DIR`):
* `Z_final.npy` – wektory C\* (24 046 × 512, `float32`, wiersze w kolejności CSV; pierwsze 256 kolumn to metoda A, kolejne 256 to B\*; każdy wiersz ma długość 1, więc iloczyn skalarny = cosinus);
* `perfume_index.csv` – `row` (numer wiersza w `Z_final.npy`), `perfume_id`, `url`, `Perfume`, `Brand` – klucz łączący wektor z perfumami;
* `embeddings_meta.json` – użyte parametry (powtarzalność).

**Dlaczego osobno wektory i indeks:** wektor to 512 liczb, a informacje o perfumach to tekst; trzymane osobno dają się niezależnie łączyć po `perfume_id` (to samo zrobisz w bazie).

In [7]:
out = CONFIG["OUTPUT_DIR"]
np.save(os.path.join(out, "Z_final.npy"), Z_final)
indeks = pd.DataFrame({"row": np.arange(N), "perfume_id": df.perfume_id, "url": df.url, "Perfume": df.Perfume, "Brand": df.Brand})
indeks.to_csv(os.path.join(out, "perfume_index.csv"), sep=";", index=False, encoding="utf-8-sig")
json.dump({"config": {k: v for k, v in CONFIG.items() if k != "INPUT_CSV_CANDIDATES"}, "kształt": list(Z_final.shape),
           "kolumny 0-255": "metoda A", "kolumny 256-511": "metoda B*"},
          open(os.path.join(out, "embeddings_meta.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)

Z_wczytane = np.load(os.path.join(out, "Z_final.npy"))
print("zapisane pliki:", sorted(os.listdir(out)))
print("kształt:", Z_wczytane.shape, "| odczyt zgodny z zapisem:", bool(np.allclose(Z_wczytane, Z_final)),
      "| perfume_id unikalne:", bool(indeks.perfume_id.is_unique))

zapisane pliki: ['Z_final.npy', 'embeddings_meta.json', 'perfume_index.csv']
kształt: (24046, 512) | odczyt zgodny z zapisem: True | perfume_id unikalne: True
